# Tarea 01: Función de Predicción y Despliegue

**Universidad / Automatización**  
**Dataset base:** [Retail Store Inventory Forecasting Dataset - Kaggle](https://www.kaggle.com/datasets/anirudhchauhan/retail-store-inventory-forecasting-dataset)

**Objetivo:** Desarrollar funciones que reciban un `product_id` y una fecha (o un lote de productos para una misma fecha) y devuelvan la predicción de demanda usando el modelo LSTM entrenado en la fase anterior.

---

## Índice

- [1. Objetivos y alcance](#1-objetivos-y-alcance)
- [2. Setup del entorno](#2-setup-del-entorno)
- [3. Carga del modelo y el escalador](#3-carga-del-modelo-y-el-escalador)
- [4. Construcción de la ventana de entrada](#4-construcción-de-la-ventana-de-entrada)
- [5. Predicción individual: `predecir_venta`](#5-predicción-individual-predecir_venta)
- [6. Predicción en lote: `predecir_batch`](#6-predicción-en-lote-predecir_batch)
- [7. Comparación con valor real](#7-comparación-con-valor-real)
- [8. Casos de error](#8-casos-de-error)
- [9. Conclusiones](#9-conclusiones)
- [10. Anexo: reproducibilidad](#10-anexo-reproducibilidad)


## 1. Objetivos y alcance

Requisitos a cumplir:

- [x] Función que recibe `product_id` y una fecha y devuelve la predicción de demanda.
- [x] Función que recibe un lote de productos para la misma fecha y devuelve la lista de predicciones.
- [x] Usar el modelo LSTM entrenado en la fase anterior (`modelos/fase2/lstm_final.keras`).
- [x] Código simple y estructurado, en el paquete `src/fase3/`.

**Flujo:**

```
producto_id + fecha
        │
        ▼
últimos 7 días de historial  →  ventana (7, 15)
        │
        ▼
escalar (0-1)  →  LSTM  →  desescalar  →  unidades reales
```


## 2. Setup del entorno

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # silencia logs de TensorFlow

import sys
from pathlib import Path

BASE = Path('..').resolve()
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

import pandas as pd

from src.fase3 import inicializar, predecir_batch, predecir_venta
from src.fase3.config import RUTA_DATOS
from src.fase3.entrada import construir_ventana, ventana_disponible

print('Entorno listo.')


Entorno listo.


## 3. Carga del modelo y el escalador

El LSTM se entrenó con las 15 variables escaladas entre 0 y 1. El escalador se carga desde `modelos/fase2/escalador.pkl` (guardado en el entrenamiento), no se reconstruye. La carga es automática la primera vez que se predice; aquí se hace explícita para dejarlo claro.

In [2]:
inicializar()
print('Modelo y escalador cargados.')
print('Ventana esperada:', 7, 'días')


I0000 00:00:1793205240.428120  591724 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1793205241.119052  591724 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Modelo y escalador cargados.
Ventana esperada: 7 días


## 4. Construcción de la ventana de entrada

Para predecir una fecha se toman los 7 días anteriores de ese producto. Cada fila contiene: `cantidad_vendida`, `es_fin_de_semana`, `es_feriado`, `dia_sin`, `dia_cos` y el one-hot del producto (15 columnas en total).

In [3]:
df = pd.read_csv(RUTA_DATOS, encoding='utf-8-sig', parse_dates=['fecha_venta'])

producto, fecha = 'P0001', '2022-03-02'
print('¿Ventana disponible?', ventana_disponible(df, producto, fecha))

X = construir_ventana(df, producto, fecha)
print('Forma:', X.shape)
print('Primera fila:', X[0])
print('Última fila :', X[-1])


¿Ventana disponible? True
Forma: (7, 15)
Primera fila: [96.          0.          0.          0.9749279  -0.22252093  1.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.        ]
Última fila : [159.          0.          1.          0.7818315   0.6234898   1.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.       ]


## 5. Predicción individual: `predecir_venta`

In [4]:
for producto in ['P0001', 'P0008']:
    print(f'{producto} 2022-03-02: {predecir_venta(producto, "2022-03-02"):.2f} unidades')


P0001 2022-03-02: 122.85 unidades
P0008 2022-03-02: 165.15 unidades


## 6. Predicción en lote: `predecir_batch`

Recibe una lista de productos para la misma fecha. Si un producto falla, se registra el error sin interrumpir el resto.

In [5]:
for r in predecir_batch(['P0001', 'P0002', 'P0003', 'P0008', 'P0010'], '2022-03-02'):
    print(r)


{'producto_id': 'P0001', 'fecha': '2022-03-02', 'prediccion': 122.85}
{'producto_id': 'P0002', 'fecha': '2022-03-02', 'prediccion': 124.32}
{'producto_id': 'P0003', 'fecha': '2022-03-02', 'prediccion': 142.16}
{'producto_id': 'P0008', 'fecha': '2022-03-02', 'prediccion': 165.15}
{'producto_id': 'P0010', 'fecha': '2022-03-02', 'prediccion': 105.45}


## 7. Comparación con valor real

Se predice una fecha del historial (2022-02-28) y se compara con la venta real registrada.

In [6]:
fecha, producto = pd.Timestamp('2022-02-28'), 'P0001'

prediccion = predecir_venta(producto, fecha)
real = float(df.loc[(df['product_id'] == producto) & (df['fecha_venta'] == fecha), 'cantidad_vendida'].iloc[0])

print(f'{producto} {fecha.date()}')
print(f'  Predicción: {prediccion:.2f}')
print(f'  Real      : {real:.2f}')
print(f'  Error     : {abs(prediccion - real):.2f} unidades ({abs(prediccion - real) / real * 100:.1f}%)')


P0001 2022-02-28
  Predicción: 133.77
  Real      : 168.00
  Error     : 34.23 unidades (20.4%)


## 8. Casos de error

- Producto inexistente.
- Fecha sin 7 días de historial previo.
- Lote mixto: los productos válidos se predicen y los inválidos se reportan.

In [7]:
for producto, fecha in [('P9999', '2022-03-02'), ('P0001', '2022-01-05')]:
    try:
        predecir_venta(producto, fecha)
    except ValueError as exc:
        print(f'{producto} {fecha} -> {exc}')

print()
for r in predecir_batch(['P0001', 'P9999', 'P0002'], '2022-03-02'):
    print(r)


P9999 2022-03-02 -> Producto 'P9999' no encontrado en los datos.
P0001 2022-01-05 -> Se requieren 7 días de historial para P0001 antes de 2022-01-05; solo hay 4.

{'producto_id': 'P0001', 'fecha': '2022-03-02', 'prediccion': 122.85}
{'producto_id': 'P9999', 'fecha': '2022-03-02', 'error': "Producto 'P9999' no encontrado en los datos."}
{'producto_id': 'P0002', 'fecha': '2022-03-02', 'prediccion': 124.32}


## 9. Conclusiones

- `predecir_venta` y `predecir_batch` quedan listas para usarse en despliegue.
- Reutilizan el modelo y el escalador entrenados antes, sin reentrenar.
- El lote tolera productos inválidos sin detener el resto.
- El modelo necesita los 7 días previos reales, por lo que sirve para predicción diaria con datos actualizados.


## 10. Anexo: reproducibilidad

```bash
.venv/bin/python -m src.fase3.ejecutar
```

**Módulos:**

- `src/fase3/modelo.py` — carga del LSTM y del escalador guardados.
- `src/fase3/entrada.py` — construcción de la ventana.
- `src/fase3/prediccion.py` — funciones públicas.
- `src/fase3/ejecutar.py` — CLI de prueba.
